In [1]:
from langchain_ollama import OllamaLLM

model = OllamaLLM(model='qwen')

In [2]:
model.invoke("What is Retrieval-Augmented Generation (RAG)?")

'Retrieval-Augmented Generation (RAG) is a technique used in natural language processing (NLP). RAG combines the power of retrieval and generative models.\n\nIn traditional machine learning, algorithms are trained on labeled data to make predictions. Retrieval models, on the other hand, use pre-computed information, such as keywords or phrases, from relevant sources to retrieve relevant information for a given task.\n\nGenerative models, on the other hand, can generate new text that is similar in style and content to the input text. Generative models are widely used in natural language processing, including text generation, machine translation, and sentiment analysis.\n\nRetrieval-Augmented Generation (RAG) combines retrieval models with generative models to create a more effective natural language processing system.'

In [4]:
from langchain_core.output_parsers import StrOutputParser
from langchain.prompts import PromptTemplate

template = """
Answer the question based only on the context below. If you can't 
answer the question, reply "I don't know".

Context: {context}

Question: {question}
"""

prompt = PromptTemplate.from_template(template)

parser = StrOutputParser()

chain = prompt | model | parser 

# chain.invoke("What is quantum physics")

In [5]:
from langchain_chroma import Chroma
from utils import get_embedding_function

CHROMA_PATH = "chroma"
DATA_PATH = "data"

db = Chroma(persist_directory=CHROMA_PATH, embedding_function=get_embedding_function())



In [6]:
def get_context(query_text: str):
    # Search the DB.
    # results = db.similarity_search_with_relevance_scores(query_text, k=5)
    results = db.similarity_search_with_score(query_text, k=5)

    # Store results as context
    context_text = "\n\n---\n\n".join([doc.page_content for doc, _score in results])
    return context_text

In [7]:
def query(query_text: str):
    # Get context from query
    context_text = get_context(query_text)

    print("Context:\n")
    print(context_text)
    print("\n#########")

    response = chain.invoke(
        {
            'context': context_text,
            'question': query_text
        }
    )
    print("\nResponse:")
    print(response)
    

In [8]:
query("Who is the main character?")

Context:

“This here young lady,” said the Gryphon, “she wants for to know your history, she do.”

“I’ll tell it her,” said the Mock Turtle in a deep, hollow tone: “sit down, both of you, and don’t speak a word till I’ve finished.”

So they sat down, and nobody spoke for some minutes. Alice thought to herself, “I don’t see how he can ever finish, if he doesn’t begin.” But she waited patiently.

“Once,” said the Mock Turtle at last, with a deep sigh, “I was a real Turtle.”

---

“You!” said the Caterpillar contemptuously. “Who are you?”

Which brought them back again to the beginning of the conversation. Alice felt a little irritated at the Caterpillar’s making such very short remarks, and she drew herself up and said, very gravely, “I think, you ought to tell me who you are, first.”

“Why?” said the Caterpillar.

Here was another puzzling question; and as Alice could not think of any good reason, and as the Caterpillar seemed to be in a very unpleasant state of mind, she turned away.



In [9]:
from utils import *

rag_model = RAGModel(model='qwen')

rag_model.query("What is the title of the story and the author's name?")

Context:

Alice’s Adventures in Wonderland

by Lewis Carroll

THE MILLENNIUM FULCRUM EDITION 3.0

Contents

CHAPTER I. Down the Rabbit-Hole CHAPTER II. The Pool of Tears CHAPTER III. A Caucus-Race and a Long Tale CHAPTER IV. The Rabbit Sends in a Little Bill CHAPTER V. Advice from a Caterpillar CHAPTER VI. Pig and Pepper CHAPTER VII. A Mad Tea-Party CHAPTER VIII. The Queen’s Croquet-Ground CHAPTER IX. The Mock Turtle’s Story CHAPTER X. The Lobster Quadrille CHAPTER XI. Who Stole the Tarts? CHAPTER XII. Alice’s Evidence

CHAPTER I. Down the Rabbit-Hole

---

“Suppose we change the subject,” the March Hare interrupted, yawning. “I’m getting tired of this. I vote the young lady tells us a story.”

“I’m afraid I don’t know one,” said Alice, rather alarmed at the proposal.

“Then the Dormouse shall!” they both cried. “Wake up, Dormouse!” And they pinched it on both sides at once.

The Dormouse slowly opened his eyes. “I wasn’t asleep,” he said in a hoarse, feeble voice: “I heard every word 

"The title of the story is Alice in Wonderland. The author's name is Lewis Carroll."

In [10]:
rag_model.query("Who is the main character?", show_context=False)


Response:
Alice is the main character.


'Alice is the main character.'

In [11]:
rag_model = RAGModel(model='llama3')

rag_model.query("Who is the main character?", show_context=False)


Response:
I don't know. The question is based on the context before CHAPTER I. Down the Rabbit-Hole.


"I don't know. The question is based on the context before CHAPTER I. Down the Rabbit-Hole."

In [12]:
rag_model.query("What is the title of the book and the author's name?", show_context=False)


Response:
The title of the book is "Alice's Adventures in Wonderland" and the author's name is Lewis Carroll.


'The title of the book is "Alice\'s Adventures in Wonderland" and the author\'s name is Lewis Carroll.'

In [13]:
rag_model.query("How many chapters are there in the book?", show_context=False)


Response:
I don't know. The provided context only contains information about the contents, title page, copyright information, and a small portion of Chapter I and Chapter X, but does not mention how many chapters are present in the book. Therefore, I cannot answer this question accurately based on the given text.


"I don't know. The provided context only contains information about the contents, title page, copyright information, and a small portion of Chapter I and Chapter X, but does not mention how many chapters are present in the book. Therefore, I cannot answer this question accurately based on the given text."

In [14]:
chain.invoke(
    {
        'context': """Content
CHAPTER I. Down the Rabbit-Hole
CHAPTER II. The Pool of Tears
CHAPTER III. A Caucus-Race and a Long Tale
CHAPTER IV. The Rabbit Sends in a Little Bill
CHAPTER V. Advice from a Caterpillar
CHAPTER VI. Pig and Pepper
CHAPTER VII. A Mad Tea-Party
CHAPTER VIII. The Queen's Croquet-Ground
CHAPTER IX. The Mock Turtle's Story
CHAPTER X. The Lobster Quadrille
CHAPTER XI. Who Stole the Tarts?
CHAPTER XII. Alice's Evidence""",
        'question': "How many chapters are there in the content?"
    }
)

'Easy one! Based on the context, I can see that there are 12 chapters. Therefore, my answer is:\n\nThere are 12 chapters in the content.'

In [9]:
teamviewer_password = "ragteamviewer2003k21"